# Laboratorio 6: Análisis de redes sociales en YouTube


## 1. Carga, comprensión e integración de los datos


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 140)

videos = pd.read_csv('data/youtube_videos.csv')
comments = pd.read_csv('data/youtube_comments.csv')

print('Videos:', videos.shape)
print('Comentarios:', comments.shape)


Videos: (293, 20)
Comentarios: (406, 17)


### 1.1 y 1.2 Unidad de observación y llaves primarias

En `youtube_videos.csv` cada fila es un video, y la llave primaria es `video_id`. Ese archivo
también trae `channel_id`, que es el identificador estable del canal (a diferencia de
`channel_name`, que puede repetirse o cambiar con el tiempo).

En `youtube_comments.csv` cada fila es un comentario principal (no incluye respuestas), y la
llave primaria es `comment_id`. La llave foránea hacia el otro archivo es `video_id`, y el
identificador del autor del comentario es `author_channel_id`, que es distinto del `channel_id`
del video (uno es quien comenta, el otro es quien publicó el video).


In [2]:
print('Video_id únicos en videos:', videos['video_id'].nunique(), 'de', len(videos), 'filas')
print('Comment_id únicos en comentarios:', comments['comment_id'].nunique(), 'de', len(comments), 'filas')
print()
print('Columnas relevantes de videos:')
print(videos[['video_id', 'channel_id', 'channel_name', 'category', 'view_count']].dtypes)
print()
print('Columnas relevantes de comentarios:')
print(comments[['comment_id', 'video_id', 'author_channel_id', 'author_name', 'reply_count']].dtypes)


Video_id únicos en videos: 293 de 293 filas
Comment_id únicos en comentarios: 406 de 406 filas

Columnas relevantes de videos:
video_id        object
channel_id      object
channel_name    object
category        object
view_count       int64
dtype: object

Columnas relevantes de comentarios:
comment_id           object
video_id             object
author_channel_id    object
author_name          object
reply_count           int64
dtype: object


### 1.3 Relación entre canal, video, autor, comentario, categoría y consulta

Un canal (`channel_id`) puede publicar varios videos, y cada video pertenece a un solo canal.
Cada video tiene una `category` asignada por YouTube y llegó al conjunto de datos a través de
una o varias consultas de búsqueda (`source_query`, `query_hits`). Un comentario pertenece a un
solo video, y fue escrito por un autor (`author_channel_id`), quien no tiene que ver con el
canal que publicó el video. Un mismo autor puede comentar en varios videos, y eso es justamente
lo que nos va a servir más adelante para armar la red autor-video: el autor y el video son los
dos tipos de nodo, y el hecho de comentar es lo que los conecta.


In [3]:
print('Canales distintos que publicaron los videos:', videos['channel_id'].nunique())
print('Autores distintos que comentaron:', comments['author_channel_id'].nunique())
print('Canales que recibieron comentarios:', comments['channel_id'].nunique())


Canales distintos que publicaron los videos: 97
Autores distintos que comentaron: 332
Canales que recibieron comentarios: 8


### 1.4 Integración por `video_id`


In [4]:
comments_full = comments.merge(
    videos,
    on='video_id',
    how='left',
    suffixes=('_comentario', '_video'),
    indicator=True,
)

print(comments_full['_merge'].value_counts())
print()
matched = (comments_full['_merge'] == 'both').sum()
print(f'{matched} de {len(comments)} comentarios pudieron asociarse con un video ({matched/len(comments):.1%}).')


_merge
both          406
left_only       0
right_only      0
Name: count, dtype: int64

406 de 406 comentarios pudieron asociarse con un video (100.0%).


Todos los comentarios lograron asociarse a un video, lo cual tiene sentido porque el archivo de
comentarios se construyó a partir de los videos del otro archivo. Sin embargo, notamos algo
importante para el resto del análisis: los 406 comentarios están repartidos en muy pocos videos.


In [5]:
videos_con_comentarios = comments['video_id'].nunique()
print(f'Videos con al menos un comentario: {videos_con_comentarios} de {len(videos)} videos totales.')
print(f'Es decir, solo el {videos_con_comentarios/len(videos):.1%} de los videos del conjunto tiene comentarios en este dataset.')


Videos con al menos un comentario: 19 de 293 videos totales.
Es decir, solo el 6.5% de los videos del conjunto tiene comentarios en este dataset.


Esto es una limitación importante que vamos a retomar más adelante: la red autor-video que
construyamos en la sección 4 solo va a tener información de esos videos con comentarios, no de
los 293 videos completos. No es que el resto de videos no tenga comentarios en YouTube, sino que
la recolección de comentarios se hizo solo para una parte de los videos.


## 2. Calidad, limpieza y preprocesamiento


### 2.1 Diagnóstico inicial de calidad


In [6]:
def diagnostico(df, nombre):
    print(f'--- {nombre} ---')
    print('Dimensiones:', df.shape)
    print()
    print('Valores nulos por columna:')
    print(df.isna().sum()[df.isna().sum() > 0])
    print()
    print('Filas duplicadas completas:', df.duplicated().sum())
    print()

diagnostico(videos, 'videos')
diagnostico(comments, 'comentarios')


--- videos ---
Dimensiones: (293, 20)

Valores nulos por columna:
published_time         13
view_count_text        13
description_snippet    25
description            26
dtype: int64

Filas duplicadas completas: 0

--- comentarios ---
Dimensiones: (406, 17)

Valores nulos por columna:
viewer_rating    406
dtype: int64

Filas duplicadas completas: 0



In [7]:
# Duplicados en las llaves primarias
print('video_id duplicados:', videos['video_id'].duplicated().sum())
print('comment_id duplicados:', comments['comment_id'].duplicated().sum())
print()

# Variables sin variabilidad (constantes)
print('Valores únicos de is_pinned:', comments['is_pinned'].unique())
print('Valores no nulos de viewer_rating:', comments['viewer_rating'].notna().sum(), 'de', len(comments))


video_id duplicados: 0
comment_id duplicados: 0

Valores únicos de is_pinned: [False]
Valores no nulos de viewer_rating: 0 de 406


`is_pinned` viene siempre en `False` en los 406 comentarios, así que no aporta nada para
diferenciar registros. `viewer_rating` está vacía en el 100% de los casos. Ninguna de las dos
sirve para el análisis.

También revisamos la consistencia entre identificadores y nombres visibles: si un mismo
`channel_id` siempre corresponde al mismo `channel_name`, y si `author_channel_id` siempre
corresponde al mismo `author_name`.


In [8]:
# Consistencia channel_id <-> channel_name (en videos)
nombres_por_canal = videos.groupby('channel_id')['channel_name'].nunique()
print('Canales con más de un channel_name distinto:', (nombres_por_canal > 1).sum())

# Consistencia author_channel_id <-> author_name (en comentarios)
nombres_por_autor = comments.groupby('author_channel_id')['author_name'].nunique()
print('Autores con más de un author_name distinto:', (nombres_por_autor > 1).sum())
print()
print(comments.groupby('author_channel_id')['author_name'].nunique().sort_values(ascending=False).head())


Canales con más de un channel_name distinto: 0
Autores con más de un author_name distinto: 0

author_channel_id
UC-HeUTT6_g-VoiWds2a4H-w    1
UC-Iul5tDYH_oiAMXQH-KaaQ    1
UC-QOpE7GOxlXcHZ8b7HbQKA    1
UC-fiZBS5Gp1eCJd9ucgWnCg    1
UC-hfX8J5JPJ-dfuakzegMLA    1
Name: author_name, dtype: int64


### 2.2 Variables que no vamos a usar o que requieren cuidado

- **`viewer_rating`**: vacía en todos los registros, la descartamos.
- **`is_pinned`**: constante en `False`, no aporta variabilidad, la descartamos del análisis
  (aunque la dejamos en el dataset original por transparencia).
- **`published_time`** (videos) y **`published_text`** (comentarios): son tiempos relativos
  ("hace 2 días", "hace 3 meses") que dependen del momento en que se recolectaron los datos, no
  son fechas exactas. Para videos ya tenemos `publish_date` en formato ISO, así que usamos esa.
  Para comentarios no existe una fecha exacta equivalente, así que cualquier análisis temporal de
  comentarios queda limitado a esas categorías relativas.
- **`dataset_sources`**: describe de qué archivo de recolección vino cada registro, es útil para
  auditoría pero no es una variable analítica en sí.
- **`description_snippet`**: es un fragmento recortado de `description`, lo tratamos como
  redundante y preferimos usar `description` completa cuando se necesite texto del video.
- **`video_title`** en el archivo de comentarios: es redundante porque se puede recuperar
  haciendo el join con `videos` por `video_id`.


### 2.3 Normalización de identificadores

No vamos a reemplazar los IDs por los nombres visibles en ningún momento del análisis
(`channel_id`, `video_id`, `comment_id` y `author_channel_id` siguen siendo las llaves). Lo único
que hacemos aquí es asegurarnos de que estén en formato texto limpio, sin espacios extra.


In [9]:
id_cols_videos = ['video_id', 'channel_id']
id_cols_comments = ['comment_id', 'video_id', 'channel_id', 'author_channel_id']

for col in id_cols_videos:
    videos[col] = videos[col].astype(str).str.strip()

for col in id_cols_comments:
    comments[col] = comments[col].astype(str).str.strip()

print('IDs normalizados (sin espacios extra).')


IDs normalizados (sin espacios extra).


### 2.4 Conversión de variables de conteo almacenadas como texto

`view_count_text` viene en el formato que muestra YouTube (por ejemplo `"2,390 vistas"`), y
`like_count_text` viene igual pero con bastantes valores vacíos (representados como espacios en
blanco cuando el comentario no tiene "me gusta" visibles). Escribimos una función para
convertirlas a número, quitando el texto ("vistas"), las comas de miles y espacios en blanco.


In [10]:
def texto_a_numero(valor):
    """Convierte conteos en formato de texto de YouTube (ej. '2,390 vistas') a entero."""
    if pd.isna(valor):
        return np.nan
    texto = str(valor).strip().lower()
    if texto == '' or texto == 'nan':
        return 0
    texto = texto.replace('vistas', '').replace('visitas', '').strip()
    texto = texto.replace(',', '').replace(' ', '')
    multiplicador = 1
    if texto.endswith('mil'):
        multiplicador = 1000
        texto = texto[:-3]
    elif texto.endswith('k'):
        multiplicador = 1000
        texto = texto[:-1]
    elif texto.endswith('m'):
        multiplicador = 1_000_000
        texto = texto[:-1]
    try:
        return int(float(texto) * multiplicador)
    except ValueError:
        return np.nan

# Validamos contra la columna view_count que ya viene calculada en el dataset
videos['view_count_calculado'] = videos['view_count_text'].apply(texto_a_numero)
con_texto = videos['view_count_text'].notna()
coinciden = (videos.loc[con_texto, 'view_count_calculado'] == videos.loc[con_texto, 'view_count']).sum()
print(f'view_count coincide con nuestro cálculo en {coinciden} de {con_texto.sum()} videos con view_count_text.')

diferencias = (videos.loc[con_texto, 'view_count'] - videos.loc[con_texto, 'view_count_calculado']).abs()
diferencias = diferencias[diferencias > 0]
print(f'En los {len(diferencias)} que no coinciden, la diferencia promedio es de {diferencias.mean():.1f} vistas.')

# like_count no viene precalculado, así que lo generamos nosotros
comments['like_count'] = comments['like_count_text'].apply(texto_a_numero)
print()
print('like_count generado, resumen:')
print(comments['like_count'].describe())


view_count coincide con nuestro cálculo en 227 de 280 videos con view_count_text.
En los 53 que no coinciden, la diferencia promedio es de 76.8 vistas.

like_count generado, resumen:
count    406.000000
mean       5.726601
std       30.661298
min        0.000000
25%        0.000000
50%        1.000000
75%        2.000000
max      405.000000
Name: like_count, dtype: float64


Nuestro cálculo a partir del texto coincide con `view_count` en la mayoría de los casos, pero no
en todos. Al revisar los que no coinciden vimos que la diferencia es siempre chica, unas cuantas
vistas de más o de menos, lo cual tiene sentido si `view_count` se recolectó en un momento un
poco distinto al de `view_count_text` y mientras tanto el conteo de YouTube siguió cambiando. No
es un error de nuestra función, es una consecuencia normal de raspar datos de una plataforma que
cambia en tiempo real. De todas formas confirma que la lógica de conversión funciona, así que la
usamos con confianza para generar `like_count`, que no venía calculado. De aquí en adelante
usamos `view_count` (numérico) y `like_count` (numérico) para cualquier análisis cuantitativo,
tal como recomienda el enunciado.


### 2.5 y 2.6 Texto original y texto limpio de los comentarios

Guardamos el texto tal cual está publicado en `texto_original`, porque lo vamos a necesitar
íntegro más adelante para el análisis de sentimiento (los modelos de sentimiento funcionan mejor
con signos de puntuación, mayúsculas y emojis originales). Para el análisis de frecuencias,
tópicos y nube de palabras construimos una versión aparte, `texto_limpio`.

Para `texto_limpio` decidimos:

- pasar todo a minúsculas.
- quitar URLs.
- separar hashtags y menciones en columnas propias (`hashtags`, `menciones`) y quitarlos del
  texto limpio para que no ensucien el conteo de palabras.
- quitar signos de puntuación y números.
- quitar stopwords en español (usamos la lista de NLTK).
- aplicar stemming en vez de lematización: no encontramos un lematizador de español confiable
  sin depender de internet, así que usamos `SnowballStemmer` de NLTK, que reduce las palabras a
  su raíz de forma automática. Documentamos esto como una limitación metodológica.
- convertir emojis a una descripción en texto (por ejemplo 😂 se convierte en algo como
  `cara_llorando_de_risa`) en vez de simplemente borrarlos, para no perder esa señal de
  sentimiento/tono.


In [11]:
import re
import string
import emoji
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

stopwords_es = set(stopwords.words('spanish'))
stemmer = SnowballStemmer('spanish')

patron_url = re.compile(r'https?://\S+|www\.\S+')
patron_hashtag = re.compile(r'#\w+')
patron_mencion = re.compile(r'@\w+')

def extraer_hashtags(texto):
    return patron_hashtag.findall(str(texto))

def extraer_menciones(texto):
    return patron_mencion.findall(str(texto))

def limpiar_texto(texto):
    texto = str(texto)
    texto = emoji.demojize(texto, language='es')
    texto = texto.lower()
    texto = patron_url.sub(' ', texto)
    texto = patron_hashtag.sub(' ', texto)
    texto = patron_mencion.sub(' ', texto)
    texto = texto.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
    texto = re.sub(r'\d+', ' ', texto)
    palabras = texto.split()
    palabras = [p for p in palabras if p not in stopwords_es and len(p) > 1]
    palabras = [stemmer.stem(p) for p in palabras]
    return ' '.join(palabras)

comments['texto_original'] = comments['text']
comments['hashtags'] = comments['text'].apply(extraer_hashtags)
comments['menciones'] = comments['text'].apply(extraer_menciones)
comments['texto_limpio'] = comments['text'].apply(limpiar_texto)

comments[['texto_original', 'texto_limpio']].head(8)


,texto_original,texto_limpio
0,Ese corrupto amigo de la vieja fiscal los teng...,corrupt amig viej fiscal verbos carcel
1,"Están jóvenes porque no buscan un trabajo, tu...",joven busc trabaj suert polic gust vand ir ves...
2,Me dejaron con ganas de demandar la ilegalidad...,dej gan demand ilegal reunion virtual maquil
3,Veremos a este mafioso de Mazariegos en la cár...,ver mafios mazarieg carcel buen tiemp sombr
4,eso es para que salga de USA por su propio pie...,salg usa propi pie aut deport
5,Imagine if they had to walk back home.,imagin if they had to walk back hom
6,buenísima investigacion :hand-purple-blue-peac...,buenisim investig hand purpl blu peac hand pur...
7,"Lleven su lonchera, sacrifiquense un poco. Y r...",llev loncher sacrifiquens reintevr diner rob


### 2.7 Efecto de la limpieza


In [12]:
vacios_antes = (comments['texto_original'].str.strip() == '').sum()
vacios_despues = (comments['texto_limpio'].str.strip() == '').sum()

dup_antes = comments['texto_original'].duplicated().sum()
dup_despues = comments['texto_limpio'].duplicated().sum()

print(f'Textos vacíos antes de limpiar: {vacios_antes}')
print(f'Textos vacíos después de limpiar: {vacios_despues}')
print()
print(f'Comentarios con texto original duplicado: {dup_antes}')
print(f'Comentarios con texto limpio duplicado: {dup_despues}')
print()

largo_antes = comments['texto_original'].str.len().mean()
largo_despues = comments['texto_limpio'].str.len().mean()
print(f'Longitud promedio del texto original: {largo_antes:.1f} caracteres')
print(f'Longitud promedio del texto limpio: {largo_despues:.1f} caracteres')


Textos vacíos antes de limpiar: 0
Textos vacíos después de limpiar: 1

Comentarios con texto original duplicado: 2
Comentarios con texto limpio duplicado: 4

Longitud promedio del texto original: 139.2 caracteres
Longitud promedio del texto limpio: 79.8 caracteres


Vemos que después de limpiar aparecen más comentarios vacíos y más duplicados que antes. Tiene
sentido: varios comentarios eran casi puro emoji, signos de puntuación o stopwords, y al quitar
todo eso quedan en blanco o se vuelven iguales a otros comentarios cortos (por ejemplo dos
comentarios que solo decían "jajaja" o puro emoji terminan como el mismo texto limpio vacío).
Para el análisis exploratorio de la siguiente sección vamos a tener en cuenta que un `texto_limpio`
vacío no significa que el comentario original estuviera vacío, sino que no le quedó contenido
después de quitar stopwords, puntuación y emojis.
